# 06 -- Evaluation and Comparison

**Purpose:** Evaluate all trained models on the held-out test set. Produces:
- Accuracy, Precision, Recall, F1, PR-AUC (primary), ROC-AUC with bootstrap CIs
- Confusion matrices, PR curves, ROC curves
- Severity-stratified recall
- Per-category confounder FP analysis with Clopper-Pearson CIs (Swimmingpool, River, Lake, Fountain)
- McNemar's test for pairwise model comparison
- Predictions CSV for downstream analysis (GradCAM, calibration, etc.)

**Compute:** Colab T4 GPU for inference; CPU for metrics and plots.

**Estimated runtime:** ~5 minutes per model.

**Script:** `scripts/evaluate.py`

**Note:** Replace `TIMESTAMP` placeholders with actual model filenames.
Run each model variant, then compare using `--compare_predictions_csv`.

In [ ]:
# Mount Google Drive and verify GPU runtime
from google.colab import drive
drive.mount('/content/drive')

import subprocess
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'T4' in gpu_info.stdout:
    print('Runtime: Colab T4 GPU')
elif 'failed' in gpu_info.stderr.lower() or gpu_info.returncode != 0:
    print('Runtime: CPU only -- switch to a GPU runtime!')
else:
    print(f'Runtime: GPU detected\n{gpu_info.stdout[:200]}')

In [ ]:
%cd /content/drive/MyDrive/imagevalidation2

In [ ]:
# --- Baseline EfficientNetB0 ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_baseline_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- Baseline ResNet50 ---
!python scripts/evaluate.py \
    --arch resnet50 \
    --model_path /content/drive/MyDrive/models/resnet50_baseline_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- EfficientNetB0 + HNM (percentile) ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_hnm_percentile_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- ResNet50 + HNM (percentile) ---
!python scripts/evaluate.py \
    --arch resnet50 \
    --model_path /content/drive/MyDrive/models/resnet50_hnm_percentile_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- EfficientNetB0 + HNM (sweep) ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_hnm_sweep_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- ResNet50 + HNM (sweep) ---
!python scripts/evaluate.py \
    --arch resnet50 \
    --model_path /content/drive/MyDrive/models/resnet50_hnm_sweep_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- EfficientNetB0 Extended Training (no HNM -- control) ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_extended_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- ResNet50 Extended Training (no HNM -- control) ---
!python scripts/evaluate.py \
    --arch resnet50 \
    --model_path /content/drive/MyDrive/models/resnet50_extended_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results

In [ ]:
# --- McNemar's test: EfficientNetB0 baseline vs HNM ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_hnm_percentile_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results \
    --compare_predictions_csv ./results/predictions/efficientnet_baseline_TIMESTAMP_predictions.csv

In [ ]:
# --- McNemar's test: EfficientNetB0 HNM vs ResNet50 baseline ---
!python scripts/evaluate.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_hnm_percentile_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir ./results \
    --compare_predictions_csv ./results/predictions/resnet50_baseline_TIMESTAMP_predictions.csv

## Per-category confounder FP analysis

Reads the predictions CSVs saved by `evaluate.py` and computes false positive rates
with **Clopper-Pearson 95 % CIs** for each of the four water-body confounder categories.

| Category | n (test) | Expected CI upper bound at 0 % FP |
|---|---|---|
| Swimmingpool | ~60 | ≤ 5.9 % |
| River | ~60 | ≤ 5.9 % |
| Lake | ~60 | ≤ 5.9 % |
| Fountain | ~60 | ≤ 5.9 % |

These bounds are sufficient for publication in *Computers & Geosciences* per
Clopper & Pearson (1934). No GPU required for this cell.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.stats import beta
from IPython.display import display

CONFOUNDER_CATS = ['Swimmingpool', 'River', 'Lake', 'Fountain']
PRED_DIR = './results/predictions'


def clopper_pearson_ci(k: int, n: int, alpha: float = 0.05):
    """Exact Clopper-Pearson confidence interval for a proportion."""
    if n == 0:
        return (0.0, 1.0)
    lo = beta.ppf(alpha / 2, k, n - k + 1) if k > 0 else 0.0
    hi = beta.ppf(1 - alpha / 2, k + 1, n - k) if k < n else 1.0
    return (lo, hi)


def confounder_fp_table(pred_csv: str) -> pd.DataFrame:
    """Compute per-category FP rates with CIs from a predictions CSV."""
    df = pd.read_csv(pred_csv)
    # filename column expected; predicted as flood when predicted_label==1
    rows = []
    for cat in CONFOUNDER_CATS:
        mask = df['filename'].str.lower().str.contains(cat.lower(), na=False)
        sub = df[mask]
        n = len(sub)
        if n == 0:
            rows.append({'category': cat, 'n_test': 0, 'n_fp': 0,
                         'fp_rate': float('nan'), 'ci_lo': float('nan'), 'ci_hi': float('nan')})
            continue
        # non-flood images predicted as flood = false positives
        n_fp = int((sub['predicted_label'] == 1).sum())
        rate = n_fp / n
        lo, hi = clopper_pearson_ci(n_fp, n)
        rows.append({
            'category': cat, 'n_test': n, 'n_fp': n_fp,
            'fp_rate': rate, 'ci_lo': lo, 'ci_hi': hi,
        })
    return pd.DataFrame(rows)


pred_csvs = sorted(glob.glob(os.path.join(PRED_DIR, '*_predictions.csv')))
if not pred_csvs:
    print(f'No predictions CSVs found in {PRED_DIR} -- run the evaluate.py cells above first.')
else:
    all_rows = []
    for csv_path in pred_csvs:
        model_name = os.path.basename(csv_path).replace('_predictions.csv', '')
        tbl = confounder_fp_table(csv_path)
        tbl.insert(0, 'model', model_name)
        all_rows.append(tbl)
        print(f'\n=== {model_name} ===')
        display(
            tbl.drop('model', axis=1)
               .style.format({
                   'fp_rate': '{:.1%}',
                   'ci_lo':   '{:.1%}',
                   'ci_hi':   '{:.1%}',
               })
               .highlight_between(subset='fp_rate', left=0.15, right=1.0,
                                   props='background-color: #ffc7c7')
        )

    # Save combined table
    combined = pd.concat(all_rows, ignore_index=True)
    out_path = './results/tables/confounder_fp_rates_all_models.csv'
    os.makedirs('./results/tables', exist_ok=True)
    combined.to_csv(out_path, index=False)
    print(f'\nCombined table saved → {out_path}')

In [ ]:
# Display all saved figures
import os
from IPython.display import Image, display

fig_dir = './results/figures'
if os.path.exists(fig_dir):
    for fname in sorted(os.listdir(fig_dir)):
        if fname.endswith('.png'):
            print(f'\n--- {fname} ---')
            display(Image(filename=os.path.join(fig_dir, fname), width=600))
else:
    print(f'Figures directory not found: {fig_dir}')